# X-ray Fracture Detection uisng CNN
**Author:** Valeria Garcia Hernandez

## Objective
Build a deep learning model to classify x-ray images as fractured or non-fractured using a Convolutional Neural Network (CNN) and compare it with a pretrained ResNet model.

In [5]:
#!pip install torch torchvision

In [39]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import torchvision.models as models

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## Data Preprocessing
### Augmentation and Resize

In [9]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(size=(224,224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

### Loading the Data

In [10]:
train_data = datasets.ImageFolder("data/processed/train", transform=train_transform)
val_data = datasets.ImageFolder("data/processed/val", transform=val_transform)
test_data = datasets.ImageFolder("data/processed/test", transform=test_transform)

In [21]:
# batch size 32 since the dataset is small
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size =32, shuffle=False)

### Class Weights (Handling Imbalance)

In [22]:
class_counts = torch.tensor([90, 202], dtype=torch.float)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
class_weights = class_weights.to(device)
print(class_weights)

tensor([0.6918, 0.3082])


## CNN MODEL

### Model Definition

In [23]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear(128*28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

### Initialization
Note that we use a weighted loss function to account for imbalance.

In [32]:
model = CNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Training + Validation

In [ ]:
epochs = 10
best_val_acc = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    # validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
        
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
        
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_acc = correct/total
    print(f"epoch {epoch+1}")
    print(f"train loss: {train_loss/len(train_loader)}")
    print(f"val loss: {val_loss/len(val_loader)}")
    print(f"val accuracy: {val_acc}")
    print("-" * 30)

    # save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best model")

We saved the best model, which reported a validation accuracy of 0.7258.

### Testing

In [34]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

CNN(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=100352, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [36]:
correct = 0
total = 0
test_loss = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total

print(f"test loss: {test_loss / len(test_loader)}")
print(f"test accuracy: {test_acc}")

test loss: 0.689414918422699
test accuracy: 0.6875


In [51]:
print(confusion_matrix(all_labels, all_preds))

[[16  4]
 [14 30]]


- The CNN achieved 68.75% accuracy. This is not better than the baseline model of assigning all observations to the majority class in the training data.
- The model showed low recall for fractured class, indicating it missed a signficant portion of fractures.
- The predictions were biased toward the non-fractured class.

This highlights the limitations of a simple CNN on this dataset.

## ResNet Model (Improved Model)

### Model Definition (Pretrained ResNet)

In [40]:
resnet_model = models.resnet18(pretrained=True)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 2)
resnet_model = resnet_model.to(device)

C:\Users\valeg\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\valeg\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\valeg/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:04<00:00, 9.89MB/s]


### Fine-tuning

In [45]:
for param in resnet_model.parameters():
    param.requires_grad = True

for param in resnet_model.fc.parameters():
    param.requires_grad = True

In [46]:
optimizer = torch.optim.Adam(resnet_model.fc.parameters(), lr=1e-4)

### Training + Validation

In [47]:
epochs = 10
best_val_acc = 0

for epoch in range(epochs):
    resnet_model.train()
    train_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = resnet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    # validation
    resnet_model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
        
            outputs = resnet_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
        
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_acc = correct/total
    print(f"epoch {epoch+1}")
    print(f"train loss: {train_loss/len(train_loader)}")
    print(f"val loss: {val_loss/len(val_loader)}")
    print(f"val accuracy: {val_acc}")
    print("-" * 30)

    # save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(resnet_model.state_dict(), "best_resnet_model.pth")
        print("Saved best model")

epoch 1
train loss: 0.6395843863487244
val loss: 0.5606394708156586
val accuracy: 0.7096774193548387
------------------------------
Saved best model
epoch 2
train loss: 0.623965185880661
val loss: 0.5587358474731445
val accuracy: 0.7258064516129032
------------------------------
Saved best model
epoch 3
train loss: 0.5864324569702148
val loss: 0.5467197895050049
val accuracy: 0.7096774193548387
------------------------------
epoch 4
train loss: 0.5960604399442673
val loss: 0.5385889112949371
val accuracy: 0.7419354838709677
------------------------------
Saved best model
epoch 5
train loss: 0.6047592282295227
val loss: 0.5323247909545898
val accuracy: 0.7580645161290323
------------------------------
Saved best model
epoch 6
train loss: 0.5962255090475083
val loss: 0.5164652913808823
val accuracy: 0.7580645161290323
------------------------------
epoch 7
train loss: 0.5526106923818588
val loss: 0.5172169506549835
val accuracy: 0.7580645161290323
------------------------------
epoch 8
t

### Testing

In [48]:
resnet_model.load_state_dict(torch.load("best_resnet_model.pth"))
resnet_model.eval()
correct = 0
total = 0
test_loss = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = resnet_model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total

print(f"test loss: {test_loss / len(test_loader)}")
print(f"test accuracy: {test_acc}")

test loss: 0.5077608972787857
test accuracy: 0.71875


In [50]:
print(confusion_matrix(all_labels, all_preds))

[[16  4]
 [14 30]]


- The ResNet model achieved 71.88% accuracy.
- It improved fracture recall.

## Comparison of Models
- CNN accuracy: 68.75%.
- **ResNet accuracy: 71.88%**.
- The ResNet model outperforms the CNN.
- ResNet improves fracture detection performance, especially recall.
- This demonstrates the effectiveness of transfer learning on small datasets.

## Conclusion
The ResNet model is selected as the final model due to its improved ability to detect bone fractures.